# 03. QA Dataset Audit

This notebook audits the final 300-pair QA dataset and links it to the overlap IAA result.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

qa = load_qa_pairs()
sample = load_sample()[['file_id', 'stratum']].rename(columns={'file_id': 'verdict_id'})
qa = qa.merge(sample, on='verdict_id', how='left')
print(f'Accepted QA pairs: {len(qa)}')
qa.head()

## Question type balance

In [ ]:
qa['question_type'].value_counts().rename_axis('question_type').to_frame('count')

In [ ]:
sns.countplot(data=qa, y='question_type', order=qa['question_type'].value_counts().index, palette='viridis')
plt.title('Question type distribution')
plt.tight_layout()

## Pairs per verdict

In [ ]:
pairs_per_verdict = qa['verdict_id'].value_counts().sort_values(ascending=False)
pairs_per_verdict.describe().round(2).to_frame('value')

In [ ]:
sns.histplot(pairs_per_verdict, bins=15, color='#2ca02c')
plt.title('Pairs per verdict')
plt.xlabel('QA pairs')
plt.tight_layout()

## Gold evidence integrity

In [ ]:
pd.DataFrame({
    'empty_gold_paragraph_lists': [int((qa['n_gold_paragraphs'] == 0).sum())],
    'mean_gold_paragraphs_per_pair': [round(float(qa['n_gold_paragraphs'].mean()), 2)],
    'unique_verdicts': [int(qa['verdict_id'].nunique())],
})

## Inter-annotator agreement

In [ ]:
iaa_path = ROOT / 'data/qa_dataset/iaa_existing_overlap/kappa_summary.json'
iaa = pd.read_json(iaa_path, typ='series')
iaa